# 01 - xBD Dataset Exploration

Explore the xBD/xView2 dataset: count images, visualize samples, analyze class distributions.

In [ ]:
import os
import sys
from pathlib import Path

# Detect Kaggle environment
IS_KAGGLE = os.path.exists('/kaggle/input')

if IS_KAGGLE:
    DATA_ROOT = Path('/kaggle/input/xview2-challenge-dataset-train-and-test')
    OUTPUT_ROOT = Path('/kaggle/working')
    # Install package if needed
    !pip install -q segmentation-models-pytorch albumentations rasterio shapely
else:
    DATA_ROOT = Path('../data/raw/xbd')
    OUTPUT_ROOT = Path('../outputs')

print(f'Kaggle: {IS_KAGGLE}')
print(f'Data root: {DATA_ROOT}')
print(f'Data exists: {DATA_ROOT.exists()}')

In [ ]:
# Add project to path
if IS_KAGGLE:
    # On Kaggle, we may need to copy the src directory
    sys.path.insert(0, '/kaggle/working')
else:
    sys.path.insert(0, str(Path('../src').resolve()))

from rescuevision.data.xbd_parser import parse_xbd_json, parse_all_labels
from rescuevision.constants import DAMAGE_CLASS_TO_ID

## 1. Count Images and Labels

In [ ]:
for split in ['train', 'hold', 'test']:
    images_dir = DATA_ROOT / split / 'images'
    labels_dir = DATA_ROOT / split / 'labels'
    
    num_images = len(list(images_dir.glob('*.png'))) if images_dir.exists() else 0
    num_labels = len(list(labels_dir.glob('*.json'))) if labels_dir.exists() else 0
    
    pre_count = len(list(images_dir.glob('*_pre_disaster.png'))) if images_dir.exists() else 0
    post_count = len(list(images_dir.glob('*_post_disaster.png'))) if images_dir.exists() else 0
    
    print(f'{split}: {num_images} images ({pre_count} pre, {post_count} post), {num_labels} labels')

## 2. Parse Sample Labels and Show Structure

In [ ]:
# Parse a few sample labels
labels_dir = DATA_ROOT / 'train' / 'labels'
images_dir = DATA_ROOT / 'train' / 'images'

sample_tiles = parse_all_labels(labels_dir, images_dir, limit=50)
print(f'Parsed {len(sample_tiles)} tiles')

# Show first tile info
if sample_tiles:
    t = sample_tiles[0]
    print(f'\nTile: {t.image_id}')
    print(f'Disaster: {t.disaster_name} ({t.disaster_type})')
    print(f'Buildings: {t.num_buildings}')
    print(f'Damage dist: {t.damage_distribution}')
    print(f'Image path: {t.image_path}')

## 3. Class Distribution Analysis

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

# Parse all train labels
all_tiles = parse_all_labels(labels_dir, images_dir)

# Count damage classes
damage_counter = Counter()
disaster_types = Counter()
buildings_per_tile = []

for tile in all_tiles:
    disaster_types[tile.disaster_type] += 1
    buildings_per_tile.append(tile.num_buildings)
    for b in tile.buildings:
        damage_counter[b.damage_label or 'un-classified'] += 1

print('Damage class distribution:')
for label, count in damage_counter.most_common():
    print(f'  {label}: {count}')

print(f'\nDisaster types: {dict(disaster_types)}')
print(f'\nBuildings per tile: min={min(buildings_per_tile)}, max={max(buildings_per_tile)}, mean={sum(buildings_per_tile)/len(buildings_per_tile):.1f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Damage class distribution
labels_list = list(damage_counter.keys())
counts = [damage_counter[l] for l in labels_list]
colors = ['green', 'yellow', 'orange', 'red', 'gray']
axes[0].barh(labels_list, counts, color=colors[:len(labels_list)])
axes[0].set_title('Damage Class Distribution')
axes[0].set_xlabel('Count')

# Disaster type distribution
types = list(disaster_types.keys())
type_counts = [disaster_types[t] for t in types]
axes[1].barh(types, type_counts)
axes[1].set_title('Disaster Type Distribution')
axes[1].set_xlabel('Number of Tiles')

# Buildings per tile histogram
axes[2].hist(buildings_per_tile, bins=50, edgecolor='black')
axes[2].set_title('Buildings per Tile')
axes[2].set_xlabel('Number of Buildings')
axes[2].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig(OUTPUT_ROOT / 'class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Visualize Sample Images with Polygons

In [ ]:
import numpy as np
from PIL import Image
from rescuevision.utils.geometry import polygon_to_mask

fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for idx, tile in enumerate(sample_tiles[:4]):
    if tile.image_path is None or not Path(tile.image_path).exists():
        continue
    
    img = np.array(Image.open(tile.image_path).convert('RGB'))
    h, w = img.shape[:2]
    
    # Create building mask
    polys = [(b.polygon, 1) for b in tile.buildings]
    mask = polygon_to_mask(polys, h, w)
    
    # Show image
    axes[0, idx].imshow(img)
    axes[0, idx].set_title(f'{tile.disaster_name}\n{tile.num_buildings} buildings')
    axes[0, idx].axis('off')
    
    # Show mask overlay
    overlay = img.copy()
    overlay[mask == 1] = [255, 0, 0]  # red overlay
    blended = (img * 0.6 + overlay * 0.4).astype(np.uint8)
    axes[1, idx].imshow(blended)
    axes[1, idx].set_title('Building overlay')
    axes[1, idx].axis('off')

plt.tight_layout()
plt.savefig(OUTPUT_ROOT / 'sample_tiles.png', dpi=150, bbox_inches='tight')
plt.show()